# CMPE 460 Mendeley Data Preprocessing

**Member 2**

This notebook prepares version 2 of the Mendeley Multi-Class Web Attack Dataset for TextCNN and BiLSTM. Run all cells from top to bottom with `data/mendeley/baseline_dataset.csv` available in the repository.

It preserves the input template from `05_mendeley_data_inspection.ipynb`, creates stratified 70% training, 10% validation, and 20% test splits with seed 42, and fits the character vocabulary on training requests.

The maximum input length is the training 99th-percentile length rounded up to a multiple of 64. The prepared arrays, source row IDs, vocabulary, and configuration are saved together in `data/mendeley/preprocessed.npz`.

[Dataset version 2](https://data.mendeley.com/datasets/zhzd5j9ffy/2) · DOI 10.17632/zhzd5j9ffy.2


## 1. Load the Source File

String loading preserves payloads and URL encoding. The file SHA-256 records the exact input bytes. If dependencies are missing, install them once using `%pip install numpy pandas scikit-learn` in a separate code cell.


In [ ]:
import hashlib
import json
import os
from itertools import combinations
from pathlib import Path
from tempfile import NamedTemporaryFile

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.model_selection import train_test_split

CSV_PATH = Path("data/mendeley/baseline_dataset.csv")
OUTPUT_PATH = Path("data/mendeley/preprocessed.npz")

if not CSV_PATH.is_file():
    raise FileNotFoundError(f"Place the baseline CSV at {CSV_PATH.resolve()}.")

raw = pd.read_csv(
    CSV_PATH,
    dtype="string",
    keep_default_na=False,
    encoding="utf-8-sig",
)
if raw.empty:
    raise ValueError("The CSV contains no data rows.")

source_sha256 = hashlib.sha256(CSV_PATH.read_bytes()).hexdigest()
print(f"Source file: {CSV_PATH.name}")
print(f"SHA-256: {source_sha256}")
print(f"Rows: {len(raw):,}")
print(f"Columns: {len(raw.columns):,}")


Source file: baseline_dataset.csv
SHA-256: d3b3509555924de6e1d37522806f2debd97ea6659e8254a95ad98d19c28a24d6
Rows: 25,736
Columns: 60


## 2. Construct Inputs and Validate the Audit

The input contains the method, URL, and payload with the same field markers used during inspection. Case, whitespace, punctuation, and URL encoding are preserved.

This baseline passed the inspection checks. The following assertions repeat the input checks before splitting. If the source file changes and fails them, inspect the affected data before preparing a new experiment.


In [ ]:
LABEL_NAMES = {
    0: "normal",
    1: "xss",
    2: "sqli",
    3: "cmdi",
    4: "lfi",
    5: "ssrf",
}
TEXT_COLUMNS = ["method", "url", "payload"]
INPUT_TEMPLATE = "[METHOD] {method}\n[URL] {url}\n[PAYLOAD] {payload}"

normalized_columns = {
    str(column).strip().casefold(): column
    for column in raw.columns
}
if len(normalized_columns) != len(raw.columns):
    raise ValueError("Column names become ambiguous after ignoring case and spaces.")

required_columns = [*TEXT_COLUMNS, "label"]
missing_columns = [name for name in required_columns if name not in normalized_columns]
if missing_columns:
    raise ValueError(f"Missing source columns: {missing_columns}.")

fields = raw[[normalized_columns[name] for name in required_columns]].copy()
fields.columns = required_columns
labels = pd.to_numeric(fields["label"], errors="coerce")
empty_inputs = pd.DataFrame({
    column: fields[column].str.strip().eq("")
    for column in TEXT_COLUMNS
}).all(axis=1)

if not labels.isin(LABEL_NAMES).all() or empty_inputs.any():
    raise ValueError("Inspect invalid labels and empty inputs using notebook 05.")

requests = (
    "[METHOD] " + fields["method"]
    + "\n[URL] " + fields["url"]
    + "\n[PAYLOAD] " + fields["payload"]
)
if requests.duplicated().any():
    raise ValueError("Repeated inputs require an explicit cleaning or grouping policy before splitting.")

data = pd.DataFrame({
    "row_id": np.arange(len(raw), dtype=np.int64),
    "request": requests,
    "label": labels.astype("int64"),
})
label_names = list(LABEL_NAMES.values())

print("Input and label checks passed.")
display(pd.Series(LABEL_NAMES, name="class").rename_axis("label_id").to_frame())


Input and label checks passed.


,class
label_id,
0,normal
1,xss
2,sqli
3,cmdi
4,lfi
5,ssrf


## 3. Create Reproducible Splits

Both splitting operations preserve class proportions. The first assigns 70% of rows to training. The remaining 30% is divided into validation and test in a one-to-two ratio. Integer rounding may shift the exact percentages slightly.

Source row IDs identify the original CSV rows. The audit checks that every row appears once and that full request texts are disjoint across splits.


In [ ]:
SEED = 42
SPLITS = ("train", "validation", "test")

train_frame, holdout_frame = train_test_split(
    data,
    train_size=0.70,
    random_state=SEED,
    stratify=data["label"],
)
validation_frame, test_frame = train_test_split(
    holdout_frame,
    test_size=2 / 3,
    random_state=SEED,
    stratify=holdout_frame["label"],
)

frames = {
    "train": train_frame.reset_index(drop=True),
    "validation": validation_frame.reset_index(drop=True),
    "test": test_frame.reset_index(drop=True),
}

source_rows = {
    split: frame["row_id"].to_numpy(dtype=np.int64)
    for split, frame in frames.items()
}
all_source_rows = np.concatenate(list(source_rows.values()))
assert np.array_equal(np.sort(all_source_rows), np.arange(len(data)))

split_summary = pd.DataFrame({
    "rows": {split: len(frame) for split, frame in frames.items()},
    "share_pct": {split: 100 * len(frame) / len(data) for split, frame in frames.items()},
}).rename_axis("split")
display(split_summary.round(3))

class_counts = pd.DataFrame({
    split: frame["label"].value_counts().reindex(list(LABEL_NAMES), fill_value=0)
    for split, frame in frames.items()
})
class_counts.index = pd.Index(label_names, name="class")
assert class_counts.gt(0).all().all()
display(class_counts)

request_sets = {split: set(frame["request"]) for split, frame in frames.items()}
overlaps = pd.DataFrame([
    {
        "split_a": first,
        "split_b": second,
        "shared_full_requests": len(request_sets[first] & request_sets[second]),
        "shared_source_rows": len(set(source_rows[first]) & set(source_rows[second])),
    }
    for first, second in combinations(SPLITS, 2)
])
assert overlaps[["shared_full_requests", "shared_source_rows"]].eq(0).all().all()
display(overlaps)


,rows,share_pct
split,,
train,18015,69.999
validation,2573,9.998
test,5148,20.003


,train,validation,test
class,,,
normal,8400,1200,2400
xss,2140,305,612
sqli,2251,321,644
cmdi,1798,257,513
lfi,2100,300,600
ssrf,1326,190,379


,split_a,split_b,shared_full_requests,shared_source_rows
0,train,validation,0,0
1,train,test,0,0
2,validation,test,0,0


## 4. Fit the Training Vocabulary and Select Input Length

The vocabulary contains characters observed in training requests. ID 0 is padding and ID 1 represents an unseen character.

The length policy uses the 99th percentile of training request lengths and rounds upward to a multiple of 64. Its observed coverage is reported during encoding. Validation and test requests use the fitted vocabulary and selected length.


In [ ]:
PAD_ID = 0
UNK_ID = 1
LENGTH_QUANTILE = 0.99
LENGTH_MULTIPLE = 64

train_lengths = frames["train"]["request"].str.len()
training_length_quantile = float(train_lengths.quantile(LENGTH_QUANTILE))
MAX_LENGTH = int(np.ceil(training_length_quantile / LENGTH_MULTIPLE) * LENGTH_MULTIPLE)

train_characters = sorted(set("".join(frames["train"]["request"])))
char_to_id = {
    char: index
    for index, char in enumerate(train_characters, start=2)
}
vocabulary_size = len(char_to_id) + 2

assert PAD_ID not in char_to_id.values()
assert UNK_ID not in char_to_id.values()

print(f"Training median length: {train_lengths.median():.1f} characters")
print(f"Training 95th-percentile length: {train_lengths.quantile(0.95):.1f} characters")
print(f"Training 99th-percentile length: {training_length_quantile:.1f} characters")
print(f"Selected maximum input length: {MAX_LENGTH}")
print(f"Vocabulary size including PAD and UNK: {vocabulary_size}")


Training median length: 133.0 characters
Training 95th-percentile length: 286.3 characters
Training 99th-percentile length: 476.0 characters
Selected maximum input length: 512
Vocabulary size including PAD and UNK: 222


## 5. Encode Requests and Check the Model Inputs

Requests are truncated on the right and padded on the right. True sequence lengths are retained for BiLSTM. SHA-256 fingerprints check whether any encoded rows are repeated or shared across splits after truncation and unknown-character mapping.


In [ ]:
def encode_requests(requests):
    encoded = np.full((len(requests), MAX_LENGTH), PAD_ID, dtype=np.int64)
    for row_index, text in enumerate(requests):
        character_ids = [
            char_to_id.get(char, UNK_ID)
            for char in text[:MAX_LENGTH]
        ]
        encoded[row_index, :len(character_ids)] = character_ids
    return encoded


X = {
    split: encode_requests(frames[split]["request"].tolist())
    for split in SPLITS
}
y = {
    split: frames[split]["label"].to_numpy(dtype=np.int64)
    for split in SPLITS
}
sequence_lengths = {
    split: np.minimum(
        frames[split]["request"].str.len().to_numpy(dtype=np.int64),
        MAX_LENGTH,
    )
    for split in SPLITS
}

audit_rows = []
encoded_fingerprints = {}

for split in SPLITS:
    assert X[split].dtype == np.int64
    assert y[split].dtype == np.int64
    assert X[split].shape == (len(y[split]), MAX_LENGTH)
    assert X[split].min() >= PAD_ID
    assert X[split].max() < vocabulary_size
    assert np.all((y[split] >= 0) & (y[split] < len(label_names)))
    assert np.array_equal(
        (X[split] != PAD_ID).sum(axis=1),
        sequence_lengths[split],
    )

    original_lengths = frames[split]["request"].str.len().to_numpy(dtype=np.int64)
    unknown_count = int((X[split] == UNK_ID).sum())
    fingerprints = [
        hashlib.sha256(row.tobytes()).digest()
        for row in X[split]
    ]
    encoded_fingerprints[split] = set(fingerprints)

    audit_rows.append({
        "split": split,
        "rows": len(y[split]),
        "input_length": MAX_LENGTH,
        "truncated_requests": int((original_lengths > MAX_LENGTH).sum()),
        "truncated_pct": 100 * float((original_lengths > MAX_LENGTH).mean()),
        "unknown_characters": unknown_count,
        "unknown_rate_pct": 100 * unknown_count / int(sequence_lengths[split].sum()),
        "repeated_encoded_rows": len(fingerprints) - len(encoded_fingerprints[split]),
    })

assert not (X["train"] == UNK_ID).any()
encoding_audit = pd.DataFrame(audit_rows).set_index("split")
display(encoding_audit.round(4))

encoded_overlaps = pd.DataFrame([
    {
        "split_a": first,
        "split_b": second,
        "shared_encoded_requests": len(
            encoded_fingerprints[first] & encoded_fingerprints[second]
        ),
    }
    for first, second in combinations(SPLITS, 2)
])
display(encoded_overlaps)
assert encoded_overlaps["shared_encoded_requests"].eq(0).all()

for split in SPLITS:
    print(f"{split}: X={X[split].shape}, y={y[split].shape}")

print("All split and encoding checks passed.")


,rows,input_length,truncated_requests,truncated_pct,unknown_characters,unknown_rate_pct,repeated_encoded_rows
split,,,,,,,
train,18015,512,148,0.8215,0,0.0000,0
validation,2573,512,22,0.8550,8,0.0021,0
test,5148,512,32,0.6216,32,0.0042,0


,split_a,split_b,shared_encoded_requests
0,train,validation,0
1,train,test,0
2,validation,test,0


train: X=(18015, 512), y=(18015,)
validation: X=(2573, 512), y=(2573,)
test: X=(5148, 512), y=(5148,)
All split and encoding checks passed.


## 6. Save and Verify the Prepared Data

One compressed NumPy archive stores the arrays and a JSON configuration. It includes labels, true sequence lengths, and source row IDs for each split. A temporary file is replaced atomically after writing so that a previous complete archive remains available if writing fails.

The two model notebooks will load this archive. The generated file belongs to the repository's ignored `data/` folder.


In [ ]:
metadata = {
    "schema_version": 1,
    "dataset_id": "zhzd5j9ffy",
    "dataset_version": 2,
    "dataset_doi": "10.17632/zhzd5j9ffy.2",
    "source_csv_sha256": source_sha256,
    "source_rows": len(data),
    "input_template": INPUT_TEMPLATE,
    "seed": SEED,
    "split_ratios": {"train": 0.70, "validation": 0.10, "test": 0.20},
    "split_sizes": {split: len(frames[split]) for split in SPLITS},
    "length_quantile": LENGTH_QUANTILE,
    "length_multiple": LENGTH_MULTIPLE,
    "training_length_quantile": training_length_quantile,
    "max_length": MAX_LENGTH,
    "pad_id": PAD_ID,
    "unk_id": UNK_ID,
    "label_names": label_names,
    "char_to_id": char_to_id,
    "vocabulary_size": vocabulary_size,
    "numpy_version": np.__version__,
    "sklearn_version": sklearn.__version__,
}

arrays_to_save = {
    "metadata_json": np.asarray(json.dumps(metadata, ensure_ascii=False)),
}
for split in SPLITS:
    arrays_to_save[f"X_{split}"] = X[split]
    arrays_to_save[f"y_{split}"] = y[split]
    arrays_to_save[f"lengths_{split}"] = sequence_lengths[split]
    arrays_to_save[f"row_ids_{split}"] = source_rows[split]

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
temporary_path = None
try:
    with NamedTemporaryFile(
        dir=OUTPUT_PATH.parent,
        prefix="preprocessed-",
        suffix=".npz",
        delete=False,
    ) as output_file:
        temporary_path = Path(output_file.name)
        np.savez_compressed(output_file, **arrays_to_save)
        output_file.flush()
        os.fsync(output_file.fileno())

    os.replace(temporary_path, OUTPUT_PATH)
finally:
    if temporary_path is not None:
        temporary_path.unlink(missing_ok=True)

with np.load(OUTPUT_PATH, allow_pickle=False) as saved:
    restored_metadata = json.loads(saved["metadata_json"].item())
    assert restored_metadata == metadata
    for split in SPLITS:
        np.testing.assert_array_equal(saved[f"X_{split}"], X[split])
        np.testing.assert_array_equal(saved[f"y_{split}"], y[split])
        np.testing.assert_array_equal(saved[f"lengths_{split}"], sequence_lengths[split])
        np.testing.assert_array_equal(saved[f"row_ids_{split}"], source_rows[split])

print(f"Prepared data saved to {OUTPUT_PATH.as_posix()}")
print(f"Archive size: {OUTPUT_PATH.stat().st_size / 1024**2:.2f} MiB")
print(f"Seed: {SEED}")
print(f"Maximum input length: {MAX_LENGTH}")
print(f"Vocabulary size: {vocabulary_size}")
print("Saved arrays and metadata verified successfully.")


Prepared data saved to data/mendeley/preprocessed.npz
Archive size: 3.17 MiB
Seed: 42
Maximum input length: 512
Vocabulary size: 222
Saved arrays and metadata verified successfully.
